# Import

## Libraries

In [20]:
import numpy as np
import pandas as pd
import pickle
from pathlib import Path

## Dataset

In [21]:
df = pd.read_csv('../Dataset/df_clean_eng.csv')
df.head()

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,cad,appet,pe,ane,classification,eGFR,comorb_score,anemia_severity,kidney_func_score,symptom_severity
0,48.0,80.0,1.020,1,0,missing,normal,notpresent,notpresent,121.000000,...,no,good,no,no,ckd,68.682456,2,-2.384153,-0.788205,0
1,7.0,50.0,1.020,4,0,missing,normal,notpresent,notpresent,140.798289,...,no,good,no,no,ckd,162.103427,0,0.807377,-1.296071,0
2,62.0,80.0,1.010,2,3,normal,normal,notpresent,notpresent,423.000000,...,no,poor,no,yes,ckd,40.838799,1,2.907140,-0.252869,2
3,48.0,70.0,1.005,4,0,normal,abnormal,present,notpresent,117.000000,...,no,poor,yes,yes,ckd,18.161458,1,2.028862,4.596228,3
4,51.0,80.0,1.010,2,0,normal,normal,notpresent,notpresent,106.000000,...,no,good,no,no,ckd,56.786414,0,0.759758,-0.964553,0


In [22]:
# Converting Object to Category
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype('category')

## Pickled Files

In [23]:
with open("../PickleFiles/ckd_imp2_artifacts.pkl", "rb") as f:
    artifacts = pickle.load(f)

In [24]:
scaler = artifacts["scaler"]
le = artifacts["label_encoder"]
feature_columns = artifacts["feature_columns"]
log_transform_cols = artifacts["log_transform_cols"]
num_cols = artifacts["num_cols"]

In [25]:
PROJECT_ROOT = Path.cwd().parent
PICKLE_DIR = PROJECT_ROOT / "PickleFiles"

In [26]:
loaded_models_v2 = {}

for pkl_file in PICKLE_DIR.glob("*_v2_hyp.pkl"):
    model_name = pkl_file.stem.replace("_v2_hyp", "")
    
    with open(pkl_file, "rb") as f:
        loaded_models_v2[model_name] = pickle.load(f)

    print(f"Loaded → {pkl_file.name}")


Loaded → Bernoulli_Naive_Bayes_v2_hyp.pkl
Loaded → Decision_Tree_v2_hyp.pkl
Loaded → Dummy_Classifier_Most_Frequent_v2_hyp.pkl
Loaded → Dummy_Classifier_Stratified_v2_hyp.pkl
Loaded → Gaussian_Naive_Bayes_v2_hyp.pkl
Loaded → Gradient_Boosting_v2_hyp.pkl
Loaded → K-Nearest_Neighbors_v2_hyp.pkl
Loaded → LightGBM_v2_hyp.pkl
Loaded → Linear_Discriminant_Analysis_v2_hyp.pkl
Loaded → Logistic_Regression_v2_hyp.pkl
Loaded → Multi-layer_Perceptron_v2_hyp.pkl
Loaded → Random_Forest_v2_hyp.pkl
Loaded → SGD_Classifier_v2_hyp.pkl
Loaded → SVM_RBF_v2_hyp.pkl
Loaded → XGBoost_v2_hyp.pkl


In [27]:
loaded_models_v2.keys()

dict_keys(['Bernoulli_Naive_Bayes', 'Decision_Tree', 'Dummy_Classifier_Most_Frequent', 'Dummy_Classifier_Stratified', 'Gaussian_Naive_Bayes', 'Gradient_Boosting', 'K-Nearest_Neighbors', 'LightGBM', 'Linear_Discriminant_Analysis', 'Logistic_Regression', 'Multi-layer_Perceptron', 'Random_Forest', 'SGD_Classifier', 'SVM_RBF', 'XGBoost'])

### How to Load a model from the above Dictionary

In [28]:
decision_tree_model = loaded_models_v2["Decision_Tree"]

In [29]:
decision_tree_model.feature_importances_

array([1., 0., 0., 0., 0.])

In [30]:
dt_params = decision_tree_model.get_params()
dt_params


{'ccp_alpha': 0.05,
 'class_weight': 'balanced',
 'criterion': 'gini',
 'max_depth': 10,
 'max_features': None,
 'max_leaf_nodes': None,
 'min_impurity_decrease': 0.05,
 'min_samples_leaf': 40,
 'min_samples_split': 40,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'random_state': 42,
 'splitter': 'best'}

# Preprocessing

In [31]:
feature_cols_imp2 = ['hemo','sg','sc','htn','bgr']

In [32]:
X = df[feature_cols_imp2].copy()
X

,hemo,sg,sc,htn,bgr
0,15.4,1.020,1.2,yes,121.000000
1,11.3,1.020,0.8,no,140.798289
2,9.6,1.010,1.8,no,423.000000
3,11.2,1.005,3.8,yes,117.000000
4,11.6,1.010,1.4,no,106.000000
...,...,...,...,...,...
391,15.7,1.020,0.5,no,140.000000
392,16.5,1.025,1.2,no,75.000000
393,15.8,1.020,0.6,no,100.000000
394,14.2,1.025,1.0,no,114.000000


In [33]:
# Log transform
for col in log_transform_cols:
    X[col] = np.log(X[col])

In [ ]:
# Scale
X[num_cols] = scaler.transform(X[num_cols])

In [35]:
# Dummy encoding
X = pd.get_dummies(X, drop_first=True)

In [36]:
# Align columns EXACTLY
X = X.reindex(columns=feature_columns, fill_value=0)


In [37]:
X

,hemo,sg,sc,bgr,htn_yes
0,1.029687,0.439451,-0.405186,-0.260689,True
1,-0.494218,0.439451,-0.863620,0.098648,False
2,-1.126080,-1.444439,0.053247,2.707146,False
3,-0.531386,-2.386384,0.898074,-0.340403,True
4,-0.382712,-1.444439,-0.230898,-0.574530,False
...,...,...,...,...,...
391,1.141192,0.439451,-1.395023,0.085165,False
392,1.438539,1.381396,-0.405186,-1.394872,False
393,1.178360,0.439451,-1.188883,-0.712701,False
394,0.583666,1.381396,-0.611326,-0.401998,False


# Explainable AI Libraries

| Person   | Assigned XAI Libraries |
|----------|------------------------|
| Person 1 | SHAP, Permutation Importance |
| Person 2 | Dalex, PDPbox |
| Person 3 | LIME, ELI5 |
| Person 4 | InterpretML, Alibi, Sklearn Inspection |


### Instructions:
- Run all the above cells
- Rename the Library headings below and add your code
- Explore each library, all the functions, plots available for all the models possible (acc to the library)
- If you need any other variable not added above, refer the previous notebooks and add those as well
- Make sure the markdown headings you add follow proper hierarchy (### for subheadings under each library)
- Once you are done, send this file in the group

## Library 1

## Library 2